In [20]:
import cv2
import numpy as np

# tensorflow와 tf.keras를 임포트
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import matplotlib.image as mping

pass_ward=[5,5,5,5]
input_pass=[]

#딥러닝
# MNIST 데이터를 학습용, 테스트 데이터로 구분하여 읽어옴
def model_load(path):
 try:
     return keras.models.load_model(path)
 except Exception:
     return None

model = model_load("mymodel.keras")
if model is not None:
    print("로드 성공")
else:
    print("로드 실패 -> 새로 생성")

    mnist = keras.datasets.mnist
    ((train_images, train_labels), (test_images, test_labels)) = mnist.load_data()

    (train_images, test_images) = (train_images / 255, test_images / 255)
#정규화 전처리

    model = keras.Sequential([ #모델구조 레이어를 위에서 아래로 쌓기 위해 필요함
   keras.layers.Input(shape=(28, 28)),
   keras.layers.Flatten(), #다음 레이어가 Dense(완전연결층)인데 Dense는 기본적으로 1차원 벡터 입력을 받는다
   keras.layers.Dense(256, activation='relu'),
    #256개 뉴런을 계산에서 feature , relu로 비선형  가중치 행렬
   keras.layers.Dense(128, activation='relu'),
   keras.layers.Dense(100, activation='relu'),

   keras.layers.Dense(10, activation='softmax')
    ])

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
#경사하강법 기본 , 손실함수 sparse_...정수(0~9) 로 들어올 때 사용 ,# metrics정확도
    model.fit(train_images, train_labels, epochs=10, verbose=1)
#학습 데이터 10번 돌아라
    test_loss, test_acc = model.evaluate(test_images, test_labels, verbose=0)
    print('\n테스트 정확도:', test_acc)
    model.save('mymodel.keras')

#tttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttt

cap = cv2.VideoCapture(0) #videocapture
if not cap.isOpened():
    print("웹캡을 열 수 업습니다.")
    exit()


while True:
    ret, frame = cap.read() #cap 읽어와서 언패킹 ret 정상적으로 읽었는지 (True/False), frame 읽어온 이미지 프레임(NumPy 배열), 보통 shape (height, width, 3) (BGR)
    if not ret:
        print("프레임을 가져 올수 없습니다.")
        break
    my_size = 100
    flip_fram =cv2.flip(frame,1) #거울
    height,width,_ = frame.shape #(세로, 가로, 채널) = (height, width, channels) (OpenCV에서 cols=가로, rows=세로 / ML feature는 데이터 열 의미)
    center_x,center_y = width//2,height//2 #반때기 중심
    roi = flip_fram[center_y -my_size :center_y+my_size,center_x-my_size:center_x+my_size] #NumPy 슬라이싱 문법 a[start:end]
#enter_y -150 부터 :center_y+150 까지 자르기
    cv2.rectangle(flip_fram,(center_x -my_size,center_y - my_size),(center_x + my_size,center_y + my_size),(0,255,0),2)
    #center_x - 150 : 중심에서 왼쪽으로 150픽셀
    #center_y - 150 : 중심에서 위로 150픽셀
    #사각형의 왼쪽 위
    #오른쪽 아래 꼭짓점

    cv2.imshow('Webcam',flip_fram)#화면 좌우반전 나오게 했음
    #화면 캡쳐를 위한 키 값 받기
    key = cv2.waitKey(1) & 0xFF # 1ms기다리며  반환값에서 하위 8비트만 쓰겠다는 처리
    if key in (ord('c') ,ord('C')): #c capture의 약자
        gray_img = cv2.cvtColor(roi,cv2.COLOR_BGR2GRAY)#그레이 컬러
        gray_img=np.flip(gray_img,1) #cv2.flip이랑 비슷
        cv2.imwrite('gray_image.png',gray_img)#이미지 저장
        gaussian_blur = cv2.GaussianBlur(gray_img,(5,5),0)#gaussianblur 5*5 커널 크기(불러 범위) 3
        # sigmaX(가우시안 표준편차) “선굵기” 아님

#이진화
        _,otsu_thread = cv2.threshold(gaussian_blur,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
        #THRESH_BINARY : 임계값 이상이면 255, 아니면 0
        #THRESH_OTSU : 임계값을 자동으로 계산
        cv2.imshow('otsu_thread',otsu_thread)

 #### Morph-----------------------------------------------
        kernel =np.ones((5,5),np.uint8) #(5,5)적당한 정도
        erosion=cv2.erode(otsu_thread,kernel,iterations = 5)
        #침식 반대 다일라이트?, 축소(침식) 흰 노이즈 제거, 침식 5번 반복이라 효과 가 강함
        cv2.imshow('erosion',erosion)
        cv2.imwrite('digit_binary.png', erosion)

        #이미지 자르기-------------------------------------------------
        img = cv2.imread('digit_binary.png', cv2.IMREAD_UNCHANGED)
        h,w = img.shape[:2] #2행 가져오기
        crop_size =280 #가로 세로 이미지를 자름
        cx,cy = int(w/2),int(h/2)
        half = crop_size // 2
        x1,x2 =cx - half, cx + half
        y1,y2 = cy - half, cy + half

        #경계면 설정-----------------------------------------------
        x1= max(0,x1)
        y1= max(0,y1)
        x2= min(w,x2)
        y2= min(h,y2)

        cropped_img = img[y1:y2,x1:x2] #슬라이싱 범위이네
        cv2.imshow('cropped_img',cropped_img)

        #이미지 반전---------------------------------------------
        reversed_img = cv2.bitwise_not(cropped_img) #bit 반전
        cv2.imshow('reversed_img',reversed_img)
        cv2.imwrite('IMAG_FOR_TEST.png', reversed_img)

         #28*28이라 축소
 #tttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttt
        # reversed_img=cv2.resize(reversed_img,(28,28)) #사이즈 줄임
        # reversed_img=reversed_img/255.0 #정규화
        # pred = model.predict(reversed_img[np.newaxis, :, :])
        # # 3차원 배열처리 #축하나를 더 쓰는 np.newaxis
        # print(pred.argmax())#확률이 큰값


        img100 = cv2.imread('IMAG_FOR_TEST.png', cv2.IMREAD_GRAYSCALE)
        img100 = cv2.resize(img100, (28, 28) )/255.0
        pred = model.predict(img100[np.newaxis, :, :])
        #print(pred.shape) #(1,10) 나오면 정상
        #모델의 마지막 레이어가 Dense(10, softmax)
        input_passin = int(pred.argmax(axis=1)[0])
        #axis=1 = “클래스(10개) 중에서 최댓값 인덱스”를 고르라는 뜻
        print(input_passin)#확률이 큰값


        input_pass.append(input_passin)#문제가 새로운값이 계속들어가는데 하나씩 들어감 그럼 4개 배열짜리로 만들어야지
        print("현재까지:", input_pass)

        if len(input_pass)==4:
         if(pass_ward == input_pass):
            print('pass') #이럼 한번만 맞아도 ok인데
         else:
            print('fail')
         input_pass.clear()

    if key ==27:
        break

cap.release()
cv2.destroyAllWindows()

로드 성공
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
3
현재까지: [3]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
3
현재까지: [3, 3]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
3
현재까지: [3, 3, 3]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
3
현재까지: [3, 3, 3, 3]
fail
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
3
현재까지: [3]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
2
현재까지: [3, 2]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
2
현재까지: [3, 2, 2]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
7
현재까지: [3, 2, 2, 7]
fail
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
7
현재까지: [7]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
7
현재까지: [7, 7]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
6
현재까지: [7, 7, 6]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
6
현재까지: [7, 7, 6, 6]
fail
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
6
현재까지: [6]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
6
현재까지: [6, 6]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
6
현재까지: [6, 6, 6]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
6
현재까지: [6, 6, 6, 6]
fail
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
6
현재까지: [6]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
6
현재까지: [6, 6]